## 차별화 포인트
- **구매 시점 예측**: 살지 안 살지를 넘어, 언제 살지(구매 주기)를 예측
- **공동 구매 패턴**: 상품을 독립적으로 보지 않고 함께 구매되는 패턴(aisle 공동구매 점수)을 반영


## 1. 데이터 로드

In [ ]:
import sys
sys.path.append('..')
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
import warnings
warnings.filterwarnings('ignore')
from utils import reduce_memory_usage

print('데이터 로드 중...')
orders       = pd.read_csv('../data/raw/orders.csv')
prior        = pd.read_csv('../data/raw/order_products__prior.csv')
train_actual = pd.read_csv('../data/raw/order_products__train.csv')
products     = pd.read_csv('../data/raw/products.csv')
aisles       = pd.read_csv('../data/raw/aisles.csv')

orders   = reduce_memory_usage(orders)
prior    = reduce_memory_usage(prior)
products = reduce_memory_usage(products)

print(f'orders  : {orders.shape}')
print(f'prior   : {prior.shape}')
print(f'train   : {train_actual.shape}')
print(f'products: {products.shape}')

## 2. 타이밍 피처 생성 (언제 살지 예측)

prior 주문 시퀀스에서 **유저별 누적 경과일**을 계산하고,  
각 (user, product) 쌍에 대해 평균 구매 간격 · 마지막 구매 이후 경과일 · overdue 비율을 도출합니다.

In [ ]:
# 2-1. prior 주문의 유저별 누적 경과일 계산
print('유저별 누적 경과일 계산 중...')

prior_orders = orders[orders['eval_set'] == 'prior'].copy()
prior_orders = prior_orders.sort_values(['user_id', 'order_number'])
prior_orders['days_since_prior_order'] = prior_orders['days_since_prior_order'].fillna(0)

# 유저의 첫 prior 주문부터 각 주문까지의 누적 경과일
prior_orders['cum_days'] = (
    prior_orders.groupby('user_id')['days_since_prior_order'].cumsum()
)

# prior_details에 타이밍 정보 결합
prior_timed = prior.merge(
    prior_orders[['order_id', 'user_id', 'order_number', 'cum_days']],
    on='order_id', how='left'
)

print(f'타이밍 정보 결합 완료: {prior_timed.shape}')
print(prior_timed[['user_id', 'product_id', 'order_number', 'cum_days']].head())

In [ ]:
# 2-2. 유저-상품별 구매 간격 통계
print('유저-상품별 구매 간격 계산 중...')

# reset_index: 명시적 복사본 생성으로 pandas 2.0+ SettingWithCopyWarning 방지
pt = prior_timed.sort_values(['user_id', 'product_id', 'order_number']).reset_index(drop=True)

# 동일 (user, product) 내 이전 구매 누적일
pt['prev_cum_days'] = pt.groupby(['user_id', 'product_id'])['cum_days'].shift(1)

# 연속 구매 간격 (일)
pt['purchase_interval'] = pt['cum_days'] - pt['prev_cum_days']

up_timing = pt.groupby(['user_id', 'product_id']).agg(
    up_avg_interval   = ('purchase_interval', 'mean'),
    up_std_interval   = ('purchase_interval', 'std'),
    up_last_cum_days  = ('cum_days', 'max'),
    up_last_order_num = ('order_number', 'max'),
    up_buy_count      = ('order_id', 'count'),
).reset_index()

user_last_prior = (
    prior_orders.groupby('user_id')['cum_days'].max()
    .reset_index().rename(columns={'cum_days': 'user_last_cum'})
)
up_timing = up_timing.merge(user_last_prior, on='user_id', how='left')

up_timing['days_since_last_buy'] = up_timing['user_last_cum'] - up_timing['up_last_cum_days']
up_timing['up_avg_interval'] = up_timing['up_avg_interval'].fillna(-1)
up_timing['up_std_interval'] = up_timing['up_std_interval'].fillna(0)

del pt
print(f'타이밍 피처 생성 완료: {up_timing.shape}')
print(up_timing.head().to_string(index=False))

In [ ]:
# 2-3. Overdue 비율(timing_ratio) 계산
# timing_ratio = (마지막 구매 후 경과일 + train 주문까지의 추가 경과일) / 평균 구매 간격
# > 1이면 평균보다 오래 지남 → 구매 overdue 상태

train_order_info = (
    orders[orders['eval_set'] == 'train'][['user_id', 'days_since_prior_order']]
    .rename(columns={'days_since_prior_order': 'train_days_since_prior'})
)

up_timing = up_timing.merge(train_order_info, on='user_id', how='left')

# (마지막 구매 이후 경과일 + train 주문까지 경과일) / 평균 구매 간격
total_gap = up_timing['days_since_last_buy'] + up_timing['train_days_since_prior'].fillna(0)
up_timing['timing_ratio'] = np.where(
    up_timing['up_avg_interval'] > 0,
    total_gap / up_timing['up_avg_interval'],
    0.0
)

up_timing = reduce_memory_usage(up_timing)
print(f'타이밍 비율 피처 추가 완료: {up_timing.shape}')
print('\n[timing_ratio 분포 (2회 이상 구매한 상품)]')
valid_tr = up_timing[up_timing['up_avg_interval'] > 0]['timing_ratio']
print(valid_tr.describe().round(3))

## 3. 공동 구매 패턴 피처 생성 (함께 사는 상품)

상품을 aisle 단위로 묶어 **Sparse Matrix 공동 구매 행렬**을 계산합니다.  
유저의 과거 구매 이력과 행렬의 내적(dot product)으로 (user, product) 단위 공동 구매 점수를 구합니다.

In [ ]:
# 3-1. Aisle × Aisle 공동 구매 행렬 (Sparse Matrix)
print('공동 구매 행렬 계산 중...')

prod_aisle   = products[['product_id', 'aisle_id']].copy()
prior_aisled = prior.merge(prod_aisle, on='product_id', how='left')
order_aisle  = prior_aisled[['order_id', 'aisle_id']].drop_duplicates()

# order/aisle → 정수 인덱스 매핑
all_order_ids = order_aisle['order_id'].unique()
all_aisle_ids = sorted(order_aisle['aisle_id'].dropna().astype(int).unique())
order_to_idx  = {o: i for i, o in enumerate(all_order_ids)}
aisle_to_idx  = {a: i for i, a in enumerate(all_aisle_ids)}

rows_m = order_aisle['order_id'].map(order_to_idx)
cols_m = order_aisle['aisle_id'].map(aisle_to_idx)
mask   = rows_m.notna() & cols_m.notna()

# 주문 × Aisle 이진 행렬
M = csr_matrix(
    (np.ones(mask.sum()), (rows_m[mask].astype(int), cols_m[mask].astype(int))),
    shape=(len(all_order_ids), len(all_aisle_ids))
)

# Aisle × Aisle 공동 구매 행렬
cooccur = (M.T @ M).toarray().astype(np.float32)
np.fill_diagonal(cooccur, 0)

# 행 정규화: 각 aisle의 총 공동구매 횟수 대비 비율
row_sums     = cooccur.sum(axis=1, keepdims=True)
cooccur_norm = np.divide(cooccur, row_sums, where=row_sums > 0, out=np.zeros_like(cooccur))

print(f'공동 구매 행렬 완성: {cooccur_norm.shape}  (aisle 수: {len(all_aisle_ids)})')

# 상위 10개 공동 구매 aisle 조합 출력
aisle_name = dict(zip(aisles['aisle_id'], aisles['aisle']))
pairs = []
for i in range(len(all_aisle_ids)):
    j = int(np.argmax(cooccur_norm[i]))
    if cooccur_norm[i, j] > 0:
        pairs.append({
            'aisle_a': aisle_name.get(all_aisle_ids[i], all_aisle_ids[i]),
            'aisle_b': aisle_name.get(all_aisle_ids[j], all_aisle_ids[j]),
            'score'  : round(float(cooccur_norm[i, j]), 4)
        })
pairs_df = pd.DataFrame(pairs).sort_values('score', ascending=False).drop_duplicates().head(10)
print('\n[aisle별 가장 강한 공동 구매 파트너 Top 10]')
print(pairs_df.to_string(index=False))

In [ ]:
# 3-2. 유저 × Aisle 구매 벡터 → 유저별 공동 구매 친화도 계산
print('유저-상품별 공동 구매 점수 계산 중...')

# 앞 셀(c0001)이 실행되지 않은 경우 원본 데이터 재로드
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix

if 'prior' not in vars():
    prior = pd.read_csv('../data/raw/order_products__prior.csv')
if 'orders' not in vars():
    orders = pd.read_csv('../data/raw/orders.csv')
if 'products' not in vars():
    products = pd.read_csv('../data/raw/products.csv')

# c0005가 실행되지 않은 경우 aisle 인덱스 및 공동구매 행렬 재계산
if 'aisle_to_idx' not in vars() or 'cooccur_norm' not in vars() or 'all_aisle_ids' not in vars():
    print('  → c0005 결과 없음: aisle 공동구매 행렬 재계산 중...')
    _prod_aisle   = products[['product_id', 'aisle_id']].copy()
    _prior_aisled = prior.merge(_prod_aisle, on='product_id', how='left')
    _order_aisle  = _prior_aisled[['order_id', 'aisle_id']].drop_duplicates()

    all_aisle_ids = sorted(_order_aisle['aisle_id'].dropna().astype(int).unique())
    _all_order_ids = _order_aisle['order_id'].unique()
    _order_to_idx  = {o: i for i, o in enumerate(_all_order_ids)}
    aisle_to_idx   = {a: i for i, a in enumerate(all_aisle_ids)}

    _rm = _order_aisle['order_id'].map(_order_to_idx)
    _cm = _order_aisle['aisle_id'].map(aisle_to_idx)
    _mk = _rm.notna() & _cm.notna()
    M_tmp = csr_matrix(
        (np.ones(int(_mk.sum())), (_rm[_mk].values.astype(int), _cm[_mk].values.astype(int))),
        shape=(len(_all_order_ids), len(all_aisle_ids))
    )
    _cooccur = (M_tmp.T @ M_tmp).toarray().astype(np.float32)
    np.fill_diagonal(_cooccur, 0)
    _rs = _cooccur.sum(axis=1, keepdims=True)
    cooccur_norm = np.divide(_cooccur, _rs, where=_rs > 0, out=np.zeros_like(_cooccur))
    del _prod_aisle, _prior_aisled, _order_aisle, M_tmp, _cooccur, _rs
    print('  → 재계산 완료')

# 유저-aisle 구매 횟수 (prior + orders + products 에서 직접 구성)
_pa = prior.merge(orders[['order_id', 'user_id']], on='order_id', how='left')
_pa = _pa.merge(products[['product_id', 'aisle_id']], on='product_id', how='left')

user_aisle   = _pa.groupby(['user_id', 'aisle_id']).size().reset_index(name='buy_count')
del _pa

all_user_ids = sorted(user_aisle['user_id'].unique())
user_to_idx  = {u: i for i, u in enumerate(all_user_ids)}

rows_u = user_aisle['user_id'].map(user_to_idx)
cols_u = user_aisle['aisle_id'].map(aisle_to_idx)
data_u = user_aisle['buy_count'].values
mask_u = rows_u.notna() & cols_u.notna()

U = csr_matrix(
    (
        data_u[mask_u.values],
        (rows_u[mask_u].values.astype(int), cols_u[mask_u].values.astype(int))
    ),
    shape=(len(all_user_ids), len(all_aisle_ids))
)

# U_cooccur[user, aisle_b] = 유저가 aisle_b와 함께 사는 패턴의 강도
U_cooccur = (U @ cooccur_norm).astype(np.float32)  # (n_users × n_aisles)

print(f'유저×Aisle 공동 구매 친화도 행렬: {U_cooccur.shape}')
print('공동 구매 점수 계산 완료')

## 4. 통합 피처 테이블 구성

In [ ]:
# 4-1. 기존 피처 테이블 로드 및 train 세트 추출
import pandas as pd
import sys
sys.path.append('..')

if 'reduce_memory_usage' not in vars():
    from utils import reduce_memory_usage

print('기존 피처 테이블 로드 중...')
# k-pick_total_v3.csv: 이미 train 세트만 포함 → eval_set 필터 불필요
data = pd.read_csv('../data/prep/k-pick_total_v3.csv')
data = reduce_memory_usage(data)

# 이후 셀 호환성: label → reordered 로 통일
if 'label' in data.columns and 'reordered' not in data.columns:
    data = data.rename(columns={'label': 'reordered'})

print(f'train 세트 크기: {data.shape}')
print(f'컬럼 목록: {data.columns.tolist()}')

In [ ]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
import sys
sys.path.append('..')

# --- 필요 변수 로드 가드 ---
if 'reduce_memory_usage' not in vars():
    from utils import reduce_memory_usage

if 'orders' not in vars():
    orders = reduce_memory_usage(pd.read_csv('../data/raw/orders.csv'))
if 'prior' not in vars():
    prior = reduce_memory_usage(pd.read_csv('../data/raw/order_products__prior.csv'))
if 'products' not in vars():
    products = reduce_memory_usage(pd.read_csv('../data/raw/products.csv'))

# up_timing 재계산 (c0003~c0004 미실행 시)
if 'up_timing' not in vars():
    print('  → up_timing 재계산 중...')
    _po = orders[orders['eval_set'] == 'prior'].copy()
    _po = _po.sort_values(['user_id', 'order_number'])
    _po['days_since_prior_order'] = _po['days_since_prior_order'].fillna(0)
    _po['cum_days'] = _po.groupby('user_id')['days_since_prior_order'].cumsum()
    _pt = prior.merge(_po[['order_id', 'user_id', 'order_number', 'cum_days']], on='order_id', how='left')
    _pt = _pt.sort_values(['user_id', 'product_id', 'order_number']).reset_index(drop=True)
    _pt['prev_cum_days'] = _pt.groupby(['user_id', 'product_id'])['cum_days'].shift(1)
    _pt['purchase_interval'] = _pt['cum_days'] - _pt['prev_cum_days']
    up_timing = _pt.groupby(['user_id', 'product_id']).agg(
        up_avg_interval   = ('purchase_interval', 'mean'),
        up_std_interval   = ('purchase_interval', 'std'),
        up_last_cum_days  = ('cum_days', 'max'),
        up_last_order_num = ('order_number', 'max'),
        up_buy_count      = ('order_id', 'count'),
    ).reset_index()
    _ulp = _po.groupby('user_id')['cum_days'].max().reset_index().rename(columns={'cum_days': 'user_last_cum'})
    up_timing = up_timing.merge(_ulp, on='user_id', how='left')
    up_timing['days_since_last_buy'] = up_timing['user_last_cum'] - up_timing['up_last_cum_days']
    up_timing['up_avg_interval'] = up_timing['up_avg_interval'].fillna(-1)
    up_timing['up_std_interval'] = up_timing['up_std_interval'].fillna(0)
    _toi = (orders[orders['eval_set'] == 'train'][['user_id', 'days_since_prior_order']]
            .rename(columns={'days_since_prior_order': 'train_days_since_prior'}))
    up_timing = up_timing.merge(_toi, on='user_id', how='left')
    _gap = up_timing['days_since_last_buy'] + up_timing['train_days_since_prior'].fillna(0)
    up_timing['timing_ratio'] = np.where(up_timing['up_avg_interval'] > 0, _gap / up_timing['up_avg_interval'], 0.0)
    up_timing = reduce_memory_usage(up_timing)
    del _po, _pt, _ulp, _toi, _gap
    print('  → up_timing 완료')

# aisle/user 인덱스 및 U_cooccur 재계산 (c0005~c0006 미실행 시)
if 'aisle_to_idx' not in vars() or 'user_to_idx' not in vars() or 'U_cooccur' not in vars():
    print('  → aisle/user 행렬 재계산 중...')
    _pa2 = prior.merge(products[['product_id', 'aisle_id']], on='product_id', how='left')
    _oa  = _pa2[['order_id', 'aisle_id']].drop_duplicates()
    all_aisle_ids  = sorted(_oa['aisle_id'].dropna().astype(int).unique())
    _all_order_ids = _oa['order_id'].unique()
    _o2i = {o: i for i, o in enumerate(_all_order_ids)}
    aisle_to_idx = {a: i for i, a in enumerate(all_aisle_ids)}
    _rm = _oa['order_id'].map(_o2i);  _cm = _oa['aisle_id'].map(aisle_to_idx)
    _mk = _rm.notna() & _cm.notna()
    _M = csr_matrix((np.ones(int(_mk.sum())),
                     (_rm[_mk].values.astype(int), _cm[_mk].values.astype(int))),
                    shape=(len(_all_order_ids), len(all_aisle_ids)))
    _co = (_M.T @ _M).toarray().astype(np.float32)
    np.fill_diagonal(_co, 0)
    _rs = _co.sum(axis=1, keepdims=True)
    cooccur_norm = np.divide(_co, _rs, where=_rs > 0, out=np.zeros_like(_co))
    del _pa2, _oa, _M, _co, _rs

    _pa3 = prior.merge(orders[['order_id', 'user_id']], on='order_id', how='left')
    _pa3 = _pa3.merge(products[['product_id', 'aisle_id']], on='product_id', how='left')
    user_aisle   = _pa3.groupby(['user_id', 'aisle_id']).size().reset_index(name='buy_count')
    del _pa3
    all_user_ids = sorted(user_aisle['user_id'].unique())
    user_to_idx  = {u: i for i, u in enumerate(all_user_ids)}
    _ru = user_aisle['user_id'].map(user_to_idx)
    _cu = user_aisle['aisle_id'].map(aisle_to_idx)
    _du = user_aisle['buy_count'].values
    _mu = (_ru.notna() & _cu.notna())
    U = csr_matrix((_du[_mu.values],
                    (_ru[_mu].values.astype(int), _cu[_mu].values.astype(int))),
                   shape=(len(all_user_ids), len(all_aisle_ids)))
    U_cooccur = (U @ cooccur_norm).astype(np.float32)
    del _ru, _cu, _du, _mu, U
    print('  → 행렬 재계산 완료')

# data 로드 (c0007 미실행 시)
if 'data' not in vars():
    data = reduce_memory_usage(pd.read_csv('../data/prep/k-pick_total_v3.csv'))
    if 'label' in data.columns and 'reordered' not in data.columns:
        data = data.rename(columns={'label': 'reordered'})

# 4-2. 타이밍 피처 결합
timing_cols = ['user_id', 'product_id',
               'up_avg_interval', 'up_std_interval',
               'days_since_last_buy', 'timing_ratio', 'up_buy_count']
data = data.merge(up_timing[timing_cols], on=['user_id', 'product_id'], how='left')

# 4-3. aisle_id 확보 (없으면 products에서 결합)
if 'aisle_id' not in data.columns:
    data = data.merge(products[['product_id', 'aisle_id']], on='product_id', how='left')

# 4-4. 공동 구매 점수 lookup
data['_user_idx']  = data['user_id'].map(user_to_idx)
data['_aisle_idx'] = data['aisle_id'].map(aisle_to_idx)

valid  = data['_user_idx'].notna() & data['_aisle_idx'].notna()
u_idxs = data.loc[valid, '_user_idx'].astype(int).values
a_idxs = data.loc[valid, '_aisle_idx'].astype(int).values

data['copurchase_score'] = 0.0
data.loc[valid, 'copurchase_score'] = U_cooccur[u_idxs, a_idxs]
data.drop(columns=['_user_idx', '_aisle_idx'], inplace=True)
data = reduce_memory_usage(data)

print(f'통합 데이터 크기: {data.shape}')
print(f'\n[추가된 차별화 피처]')
new_feats = ['up_avg_interval', 'up_std_interval', 'days_since_last_buy',
             'timing_ratio', 'up_buy_count', 'copurchase_score']
print(data[new_feats].describe().round(3).to_string())

## 5. 모델 1 – 구매 시점 예측 (언제 살까?)

`up_avg_interval`(유저-상품별 평균 구매 간격)을 기준으로 **4개 구매 주기 버킷**을 예측합니다.  
2회 이상 구매 이력이 있어 실제 간격을 알 수 있는 샘플로 학습하고,  
1회 구매 또는 신규 상품에 대해 구매 주기를 **추정**하는 것이 모델의 차별화 가치입니다.

In [ ]:
# 5-1. 구매 시점 버킷 레이블 생성
# 0: 매우 빈번 (≤ 7일)   - 주 1회 이상
# 1: 빈번     (8-15일)   - 격주
# 2: 보통     (16-30일)  - 월 1회
# 3: 드물게   (> 30일)   - 월 1회 미만

# 2회 이상 구매(avg_interval > 0)인 케이스만 사용
timing_data = data[data['up_avg_interval'] > 0].copy()

# pd.cut() 벡터화: apply() Python 루프 대신 numpy 기반으로 수백만 행 처리
timing_data['timing_label'] = pd.cut(
    timing_data['up_avg_interval'],
    bins=[0, 7, 15, 30, float('inf')],
    labels=[0, 1, 2, 3],
    right=True
).astype(int)

label_names = {0: '매우빈번(≤7일)', 1: '빈번(8-15일)', 2: '보통(16-30일)', 3: '드물게(>30일)'}
print('[구매 시점 버킷 분포]')
for k in range(4):
    cnt = (timing_data['timing_label'] == k).sum()
    print(f'  {k} {label_names[k]:15s}: {cnt:>10,}건  ({cnt/len(timing_data)*100:.1f}%)')

In [ ]:
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# 5-2. 타이밍 모델 데이터 준비
unused_t = ['user_id', 'order_id', 'eval_set', 'product_id', 'reordered',
            'order_number', 'up_avg_interval', 'timing_label']
feat_cols_t = [c for c in timing_data.columns
               if c not in unused_t and timing_data[c].dtype != object]

X_t = timing_data[feat_cols_t].fillna(0)
y_t = timing_data['timing_label'].astype(int)

X_trv, X_te_t, y_trv, y_te_t = train_test_split(X_t, y_t, test_size=0.1, random_state=42, stratify=y_t)
X_tr_t, X_va_t, y_tr_t, y_va_t = train_test_split(X_trv, y_trv, test_size=2/9, random_state=42, stratify=y_trv)

total = len(y_t)
print('[데이터 분리 결과 (7:2:1)]')
print(f'  학습: {len(y_tr_t):>10,}  검증: {len(y_va_t):>10,}  테스트: {len(y_te_t):>10,}')

dtrain_t = lgb.Dataset(X_tr_t, label=y_tr_t)
dval_t   = lgb.Dataset(X_va_t, label=y_va_t, reference=dtrain_t)

params_t = {
    'objective'      : 'multiclass',
    'num_class'      : 4,
    'metric'         : 'multi_logloss',
    'boosting_type'  : 'gbdt',
    'learning_rate'  : 0.05,
    'num_leaves'     : 31,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq'   : 5,
    'seed'           : 42,
    'verbose'        : -1
}

print('\n[구매 시점 예측 모델 학습 시작]')
model_timing = lgb.train(
    params_t, dtrain_t, num_boost_round=500,
    valid_sets=[dval_t], valid_names=['valid'],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)]
)
print(f'\n최적 반복 횟수: {model_timing.best_iteration}')

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 5-3. 구매 시점 예측 성능 평가
probs_t = model_timing.predict(X_te_t)
preds_t = probs_t.argmax(axis=1)

print('[구매 시점 예측 모델 성능]')
print(f'  Accuracy: {accuracy_score(y_te_t, preds_t):.4f}')
print()
print(classification_report(
    y_te_t, preds_t,
    target_names=['매우빈번(≤7일)', '빈번(8-15일)', '보통(16-30일)', '드물게(>30일)'],
    zero_division=0
))

# 피처 중요도 시각화
imp_t = pd.DataFrame({
    'Feature': feat_cols_t,
    'Gain'   : model_timing.feature_importance(importance_type='gain')
}).sort_values('Gain', ascending=False).head(15)

plt.figure(figsize=(10, 6))
plt.title('구매 시점 예측 모델 - 피처 중요도 (상위 15개)')
sns.barplot(x='Gain', y='Feature', data=imp_t, palette='mako')
for i, v in enumerate(imp_t['Gain']):
    plt.text(v, i, f' {v:,.0f}', va='center', fontsize=8)
plt.tight_layout()
plt.show()

## 6. 모델 2 – 공동구매 강화 재구매 예측 (살지 말지 + 언제 살지)

기존 피처에 **타이밍 피처** + **공동 구매 점수**를 추가하여 재구매 예측 성능을 향상시킵니다.

In [ ]:
# 6-1. 재구매 예측 모델 데이터 준비
unused2 = ['user_id', 'order_id', 'eval_set', 'product_id', 'reordered', 'order_number']
feat_cols2 = [c for c in data.columns
              if c not in unused2 and data[c].dtype != object]

X2 = data[feat_cols2].fillna(0)
y2 = data['reordered'].astype(int)

X_trv2, X_te2, y_trv2, y_te2 = train_test_split(X2, y2, test_size=0.1, random_state=42, stratify=y2)
X_tr2, X_va2, y_tr2, y_va2   = train_test_split(X_trv2, y_trv2, test_size=2/9, random_state=42, stratify=y_trv2)

total2 = len(y2)
print('[데이터 분리 결과 (7:2:1)]')
print(f'  학습 : {len(y_tr2):>10,}건  ({len(y_tr2)/total2:.1%})')
print(f'  검증 : {len(y_va2):>10,}건  ({len(y_va2)/total2:.1%})')
print(f'  테스트: {len(y_te2):>10,}건  ({len(y_te2)/total2:.1%})')
print(f'\n사용 피처 수: {len(feat_cols2)}')
print(f'피처 목록: {feat_cols2}')

In [ ]:
# 6-2. 재구매 예측 모델 학습
scale_w2 = round((y_tr2 == 0).sum() / (y_tr2 == 1).sum(), 2)
dtrain2  = lgb.Dataset(X_tr2, label=y_tr2)
dval2    = lgb.Dataset(X_va2, label=y_va2, reference=dtrain2)

params2 = {
    'objective'        : 'binary',
    'metric'           : 'auc',
    'boosting_type'    : 'gbdt',
    'scale_pos_weight' : scale_w2,
    'learning_rate'    : 0.05,
    'num_leaves'       : 63,
    'feature_fraction' : 0.8,
    'bagging_fraction' : 0.8,
    'bagging_freq'     : 5,
    'min_data_in_leaf' : 100,
    'seed'             : 42,
    'verbose'          : -1
}

model2 = lgb.train(
    params2, dtrain2, num_boost_round=1000,
    valid_sets=[dtrain2, dval2], valid_names=['train', 'valid'],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(50)]
)
print(f'\n최적 반복 횟수: {model2.best_iteration}')
print(f'최고 검증 AUC : {model2.best_score["valid"]["auc"]:.6f}')

In [ ]:
from sklearn.metrics import f1_score, confusion_matrix, classification_report, accuracy_score

# 6-3. 최적 임계값 탐색 및 성능 평가
test_probs2 = model2.predict(X_te2)
best_f1_2, best_t2 = 0, 0.5

print(f"{'임계값':<10} | {'F1-Score':<10}")
print('-' * 25)
for t in np.arange(0.1, 0.91, 0.05):
    preds = (test_probs2 >= t).astype(int)
    f1 = f1_score(y_te2, preds)
    print(f'{t:<10.2f} | {f1:<10.4f}')
    if f1 > best_f1_2:
        best_f1_2, best_t2 = f1, t
print('-' * 25)
print(f'최적 임계값: {best_t2:.2f}  |  최고 F1-Score: {best_f1_2:.4f}')

In [ ]:
# 6-4. 최종 성능 리포트
final_preds2 = (test_probs2 >= best_t2).astype(int)
tn, fp, fn, tp = confusion_matrix(y_te2, final_preds2).ravel()

sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)
precision   = tp / (tp + fp)
npv         = tn / (tn + fn)
prevalence  = (tp + fn) / (tp + tn + fp + fn)
det_rate    = tp / (tp + tn + fp + fn)
det_prev    = (tp + fp) / (tp + tn + fp + fn)
bal_acc     = (sensitivity + specificity) / 2

print('=' * 60)
print(f'  차별화 모델 최종 성능 리포트 (임계값: {best_t2:.2f})')
print('=' * 60)
print(f'  Accuracy             : {accuracy_score(y_te2, final_preds2):.4f}')
print(f'  F1-Score             : {best_f1_2:.4f}')
print()
print(f'  Sensitivity (Recall) : {sensitivity:.5f}  <- 재구매자 중 맞춘 비율')
print(f'  Specificity          : {specificity:.5f}  <- 미구매자 중 맞춘 비율')
print(f'  Pos Pred Value (PPV) : {precision:.5f}  <- 산다 예측 중 실제 구매 비율')
print(f'  Neg Pred Value (NPV) : {npv:.5f}')
print(f'  Prevalence           : {prevalence:.5f}')
print(f'  Detection Rate       : {det_rate:.5f}')
print(f'  Detection Prevalence : {det_prev:.5f}')
print(f'  Balanced Accuracy    : {bal_acc:.5f}')
print('-' * 60)
print()
print('[기본 분류 리포트]')
print(classification_report(y_te2, final_preds2, target_names=['미구매(0)', '재구매(1)']))

## 7. 피처 중요도 및 차별화 피처 기여도 분석

In [ ]:
# 7-1. 피처 중요도 (신규 피처 하이라이트)
imp2 = pd.DataFrame({
    'Feature': feat_cols2,
    'Gain'   : model2.feature_importance(importance_type='gain'),
    'Split'  : model2.feature_importance(importance_type='split'),
}).sort_values('Gain', ascending=False).reset_index(drop=True)

new_features = ['up_avg_interval', 'up_std_interval', 'days_since_last_buy',
                'timing_ratio', 'up_buy_count', 'copurchase_score']
imp2['is_new'] = imp2['Feature'].isin(new_features)

print('[피처별 Gain (재구매 예측 기여도 내림차순)]')
print(imp2.to_string(index=False))

print('\n[차별화 신규 피처 중요도]')
print(imp2[imp2['is_new']].to_string(index=False))

In [ ]:
# 7-2. 피처 중요도 시각화 (신규 vs 기존 피처 구분)
# .tolist(): pandas Series를 matplotlib color에 전달 시 버전에 따른 TypeError 방지
colors = imp2['is_new'].map({True: '#e74c3c', False: '#3498db'}).tolist()

fig, ax = plt.subplots(figsize=(12, 8))
ax.barh(imp2['Feature'], imp2['Gain'], color=colors)
ax.set_xlabel('Importance (Total Gain)')
ax.set_title('피처 중요도 (빨간색=차별화 신규 피처, 파란색=기존 피처)')
ax.invert_yaxis()

import matplotlib.patches as mpatches
legend_handles = [
    mpatches.Patch(color='#e74c3c', label='신규 피처 (타이밍 + 공동구매)'),
    mpatches.Patch(color='#3498db', label='기존 피처')
]
ax.legend(handles=legend_handles, loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# 7-3. 공동 구매 패턴 시각화 – 상위 15개 aisle 쌍
sym_pairs = []
for i in range(len(all_aisle_ids)):
    for j in range(i + 1, len(all_aisle_ids)):
        s = float((cooccur_norm[i, j] + cooccur_norm[j, i]) / 2)
        if s > 0:
            sym_pairs.append({
                'aisle_a': aisle_name.get(all_aisle_ids[i], str(all_aisle_ids[i])),
                'aisle_b': aisle_name.get(all_aisle_ids[j], str(all_aisle_ids[j])),
                'score'  : s
            })

sym_df = pd.DataFrame(sym_pairs).sort_values('score', ascending=False).head(15)

y_labels = [
    f"{r['aisle_a'][:22]}\n+ {r['aisle_b'][:22]}"
    for _, r in sym_df.iterrows()
]

plt.figure(figsize=(12, 8))
plt.title('상위 15개 공동 구매 Aisle 조합 (높을수록 함께 구매 빈도가 높음)')
plt.barh(range(len(sym_df)), sym_df['score'].values, color='#2ecc71')
plt.yticks(range(len(sym_df)), y_labels, fontsize=8)
plt.xlabel('Co-purchase Score (Normalized)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print('\n[상위 15개 공동 구매 패턴]')
print(sym_df.to_string(index=False))

In [ ]:
# 7-4. timing_ratio 구간별 실제 재구매율 분석
# timing_ratio가 높을수록 실제로 더 많이 사는지 검증
analysis = data[data['timing_ratio'] > 0].copy()
analysis['timing_ratio_bin'] = pd.cut(
    analysis['timing_ratio'],
    bins=[0, 0.5, 1.0, 1.5, 2.0, float('inf')],
    labels=['0-0.5 (너무 이름)', '0.5-1.0 (적당)', '1.0-1.5 (약간 overdue)', '1.5-2.0 (overdue)', '>2.0 (많이 overdue)']
)

ratio_analysis = (
    analysis.groupby('timing_ratio_bin', observed=True)['reordered']
    .agg(['mean', 'count'])
    .rename(columns={'mean': '실제_재구매율', 'count': '샘플수'})
    .reset_index()
)

print('[timing_ratio 구간별 실제 재구매율]')
print('(timing_ratio가 높을수록 재구매율이 높아야 피처가 유효함)')
print(ratio_analysis.to_string(index=False))

plt.figure(figsize=(10, 5))
plt.bar(range(len(ratio_analysis)), ratio_analysis['실제_재구매율'], color='#9b59b6')
plt.xticks(range(len(ratio_analysis)), ratio_analysis['timing_ratio_bin'], rotation=15, ha='right')
plt.ylabel('실제 재구매율')
plt.title('Timing Ratio 구간별 실제 재구매율 (높을수록 overdue → 구매 확률 ↑)')
plt.tight_layout()
plt.show()